<a href="https://colab.research.google.com/github/Hunterashcraft04/SemanticsOfDataViz_HAshcraftGTUgThesis/blob/main/DataVizThesis_StimGen/DataVizStim(figure)Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate figures (.png) (scatter plot, line plot, bar graph, violin plot) + excel file (.xlsx) with the figure names

In [1]:
# import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import io
import zipfile
from google.colab import files
from scipy.spatial.distance import mahalanobis


In [3]:
# seed
seed = 1
np.random.seed(seed)

# initizations
nFigs = 72 # the number of figures wanted
dataPnts = 250
fldrName = (f"stimSetPilot_{seed}")
imgFldrName = (f"dataVizSetPilot_{seed}")
xlsxName = f"{fldrName}.xlsx"
figW = 2.44
figL = 2.44
figDPI = 300
imgDir = os.path.join(fldrName, imgFldrName)
os.makedirs(imgDir, exist_ok=True)
fns = []

if nFigs % 12 != 0: # replace the number with the amount of visualization types being used
    raise Exception("nFigs must be divisable by 12") # replace the number with the amount of visualization types being used
else:
    j = nFigs
    print(f"This will run {j} times")

if j % 2 != 0:
    raise Exception("j must be divisable by 2 to have exactly half of the figures contain outliers")

# multivariate normal distribution (https://www.sciencedirect.com/topics/biochemistry-genetics-and-molecular-biology/multivariate-normal-distribution)
# defines means; mean of x, mean of y
means = [0, 0]

# generation loop
runs = np.linspace(1, -1, j)

# outlier logic
outlierRuns = np.array([True] * (j // 2) + [False] * (j // 2))
np.random.shuffle(outlierRuns)

for i, n in enumerate(runs):

  # outlier logic
  OUTLIER = outlierRuns[i]
  outlierTREND = ""

  # covariance matrix: unit variance for x and y, zero covariance
  covMat = np.array([[1, 0],
                    [0, 1]])

  # data generation
  data = np.random.multivariate_normal(means, covMat, dataPnts)
  x = data[:, 0]
  y = data[:, 1]

  # Gram-Schmidt Orthogonalization, forces the correlation to be what 'n' is set to from the "noisy" covariance matrix data (https://www.sciencedirect.com/topics/engineering/gram-schmidt-orthogonalization)
  x = (x - np.mean(x)) / np.std(x)
  y = y - np.mean(y) # force the data to be centered and normalized
  y = y - (np.dot(x, y) / np.dot(x, x)) * x
  y = y / np.std(y) # strip accidental correlation to make z perfectly perpendicular to x

  if np.abs(n) >= 1.0:
    y = np.sign(n) * x
  else:
    y = n * x + np.sqrt(1 - n**2) * y # blends them to get the exact correlation 'n' while still perserving the "noisy" covariance matrix data

  # outlier generation
  if OUTLIER:
      xstd = np.std(x)
      ystd = np.std(y)

      # Mahalanobis distance; quantification of outlier (https://www.sciencedirect.com/topics/engineering/mahalanobis-distance)
      pts = np.column_stack((x, y))
      cntr = np.array([np.mean(x),np.mean(y)])
      covM = np.cov(pts, rowvar=False)
      invCovM = np.linalg.pinv(covM)
      disXY = np.array([
      mahalanobis(pt, cntr, invCovM)
      for pt in pts
      ])

      while True:
          # Cauchy distribution; generates a candidate outlier in the data (https://www.sciencedirect.com/topics/engineering/cauchy-distribution)
          cauchy = np.abs(np.random.standard_cauchy())
          theta = np.random.uniform(0, 2 * np.pi)
          xOutlier = np.mean(x) + (xstd * cauchy * np.cos(theta))
          yOutlier = np.mean(y) + (ystd * cauchy * np.sin(theta))
          MD = mahalanobis(
              np.array([xOutlier, yOutlier]),
              cntr, invCovM
              )

          # z-score of Mahalanobis distance
          z = (MD - np.mean(disXY)) / np.std(disXY, ddof=1)

          # test final correlation after adding the candidate outlier
          xTest = np.append(x, xOutlier)
          yTest = np.append(y, yOutlier)
          rTest = np.corrcoef(xTest, yTest)[0, 1]

          if 10 >= z >= 6 and np.sign(rTest) == np.sign(n): # z-score bounds indicative of an outlier and final correlation must retain intended sign
              break

      # correct answer/naming logic for outliers
      outlierTREND = (f"_out(MD={MD:.3f},z={z:.3f})")

  else:
      while True:
          lastPnt = np.random.multivariate_normal(means, covMat, 1)
          xOutlier, yOutlier = lastPnt[0, 0], lastPnt[0, 1]

          # test final correlation after adding the candidate point
          xTest = np.append(x, xOutlier)
          yTest = np.append(y, yOutlier)
          rTest = np.corrcoef(xTest, yTest)[0, 1]

          if np.sign(rTest) == np.sign(n):
              break

  x = np.append(x, xOutlier)
  y = np.append(y, yOutlier)

  # correct answer/naming logic for correlation
  r = np.corrcoef(x, y)[0, 1]
  if .7 > r > 0:
    corTREND = (f"_posCor(r={r:.3f})")
  elif r >= .7:
    corTREND = (f"_posSrngCor(r={r:.3f})")
  elif -.7 < r < 0:
    corTREND = (f"_negCor(r={r:.3f})")
  elif r <= -.7:
    corTREND = (f"_negSrngCor(r={r:.3f})")
  else:
    corTREND = (f"_noCor(r={r:.3f})") # this shouldn't/can't really happen

  # plot them
  # scatter plot
  plt.clf()
  plt.figure(figsize=(figW, figL), dpi=figDPI)
  plt.scatter(x, y, s=5)
  plt.xticks([]) # no tickmarks and numbers on the x axis
  plt.yticks([]) # no tickmarks and numbers on the y axis
  plt.gca().spines['top'].set_visible(False)
  plt.gca().spines['right'].set_visible(False)
  scatter = f"scatter_plot{i+1}{corTREND}{outlierTREND}.png"
  plt.savefig(f"{imgDir}/{scatter}")
  plt.close()
  fns.extend([scatter])
  print(f"\rRun {i + 1} is complete", end="", flush=True)

# zip, download, and clear
df = pd.DataFrame(fns, columns=["stim_fns"])
df.to_excel(f"{fldrName}/{xlsxName}", index=False)
shutil.make_archive(fldrName, "zip", fldrName)
print()
print(f"Now saving as {fldrName}.zip")
files.download(f"{fldrName}.zip")


This will run 72 times
Run 72 is complete
Now saving as stimSetPilot_1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<Figure size 640x480 with 0 Axes>